# Cycle 2 — Hyperparameter Tuning: xG Model

This notebook finds the optimal settings XGBoost and Random Forest and LightGBM. The scoring metric is AUC-ROC - not accuracy. The tuner will search for the combination that maximises AUC, which is the most suitable for an imbalanced binary classification problem. 

In [1]:
import sys, os               
import pandas as pd          
import numpy as np           
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler          # z-score scaler
from sklearn.ensemble import RandomForestClassifier       # Random Forest
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBClassifier                         # XGBoost
from lightgbm import LGBMClassifier                       # LightGBM
from sklearn.linear_model import LogisticRegression       # imported for reference baseline
from sklearn.dummy import DummyClassifier                 # imported for reference baseline
import joblib, os              


_here = os.getcwd()                
while not os.path.isdir(os.path.join(_here, 'data')):  
    _p = os.path.dirname(_here)                        
    if _p == _here: raise RuntimeError('project root not found')  
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)                             

from config import Paths, ensure_dirs  
ensure_dirs() 


In [7]:
df = pd.read_csv(str(Paths.WYSCOUT_PROCESSED))  # load preprocessed shot dataset
X = df.drop(columns=['Goal'])   # 9 features
y = df['Goal']                  # binary target: 0=no goal, 1=goal

# Stratified split preserves 10% goal rate in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()                      # z-score scaler
X_train_s = scaler.fit_transform(X_train)       # fit on train only — no leakage
X_test_s  = scaler.transform(X_test)            # apply same transform to test

neg, pos = y_train.value_counts()[0], y_train.value_counts()[1]  # count non-goals vs goals
scale_pos_weight = neg / pos  # imbalance ratio: ~8.25 — goals are 8× rarer than non-goals

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # inner CV for all searches

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')
print(f'CV: StratifiedKFold(n_splits=5)')


Train: 6,760 | Test: 1,691
scale_pos_weight: 8.25
CV: StratifiedKFold(n_splits=5)


## Tune XGBoost

Runs `RandomizedSearchCV` with 50 combinations over an 8-parameter XGBoost grid, evaluated via 5-fold stratified CV on the training set. Scoring metric is AUC-ROC (not accuracy — imbalanced dataset).

- XGBoost was the 3rd-best baseline (AUC 0.7871). Tuning typically yields +2–5% AUC improvement on tabular data.

- **Key addition vs Cycle 1:** `scale_pos_weight` is also tuned — we test values at 0.5×, 1×, and 1.5× the computed ratio to find the optimal class-weight balance.


In [8]:
xgb_param_grid = {
    'n_estimators':     [100, 200, 300, 500],    # boosting rounds
    'max_depth':        [3, 4, 5, 6],             # tree depth
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],  # shrinkage per tree
    'subsample':        [0.7, 0.8, 1.0],          # row subsampling
    'colsample_bytree': [0.7, 0.8, 1.0],          # feature subsampling
    'min_child_weight': [1, 3, 5],                # min samples per leaf
    'gamma':            [0, 0.1, 0.2],            # min loss reduction for split
    'scale_pos_weight': [scale_pos_weight,        # computed ratio (~8.25)
                         scale_pos_weight * 0.5,  # softer weight
                         scale_pos_weight * 1.5], # harder weight
}

total = 4*4*4*3*3*3*3*3  # all possible combinations
print(f'Total possible combinations: {total:,}')
print(f'We will try: 50 (RandomizedSearch x 5-fold CV = 250 fits)')


Total possible combinations: 15,552
We will try: 50 (RandomizedSearch x 5-fold CV = 250 fits)


In [9]:
xgb = XGBClassifier(random_state=42, eval_metric='auc', verbosity=0)  # base model

search_xgb = RandomizedSearchCV(
    xgb, xgb_param_grid,
    n_iter=50,            # 50 random combinations
    cv=cv,                # 5-fold stratified CV
    scoring='roc_auc',    # optimise AUC (not accuracy) — correct for imbalanced classes
    random_state=42,      # reproducible sampling
    n_jobs=-1,            # use all CPU cores
    verbose=1             # print progress
)
search_xgb.fit(X_train_s, y_train)  # run the search

print('Best hyperparameters:')
for k, v in search_xgb.best_params_.items():
    print(f'  {k}: {v}')
print()
print(f'Best CV AUC:   {search_xgb.best_score_:.4f}')  # best cross-validation AUC

y_prob_xgb_tuned = search_xgb.best_estimator_.predict_proba(X_test_s)[:, 1]  # goal probabilities
y_pred_xgb_tuned = search_xgb.best_estimator_.predict(X_test_s)              # class labels
print(f'Test AUC:      {roc_auc_score(y_test, y_prob_xgb_tuned):.4f}')       # held-out test AUC
print(f'Test Accuracy: {accuracy_score(y_test, y_pred_xgb_tuned)*100:.2f}%')


Fitting 5 folds for each of 50 candidates, totalling 250 fits


Best hyperparameters:
  subsample: 0.7
  scale_pos_weight: 8.247606019151847
  n_estimators: 200
  min_child_weight: 5
  max_depth: 4
  learning_rate: 0.01
  gamma: 0
  colsample_bytree: 0.7

Best CV AUC:   0.8137
Test AUC:      0.8183
Test Accuracy: 73.39%


### Observations:
- **Test AUC 0.8183** — a jump of +0.0312 from untuned XGBoost (0.7871)
- Low learning rate (0.01) + 200 trees again — same pattern as Cycle 1
- scale_pos_weight stayed at the computed 8.25 — the default imbalance ratio was already optimal
- min_child_weight=5 (conservative) — prevents overfitting on the minority Goal class
- Test AUC (0.8183) slightly above CV AUC (0.8137) — model generalises well

### Improvement over baseline:
- Untuned XGBoost:  AUC 0.7871
- Tuned XGBoost:    AUC 0.8183
- **Gain: +0.0312**

In [10]:
print('TUNED XGBOOST — Full Report')
print(f'AUC-ROC:  {roc_auc_score(y_test, y_prob_xgb_tuned):.4f}')         # primary metric
print(f'Accuracy: {accuracy_score(y_test, y_pred_xgb_tuned)*100:.2f}%')    # secondary
print()
print(classification_report(y_test, y_pred_xgb_tuned, target_names=['No Goal', 'Goal']))
# Check goal recall — are we finding most actual goals?


TUNED XGBOOST — Full Report
AUC-ROC:  0.8183
Accuracy: 73.39%

              precision    recall  f1-score   support

     No Goal       0.96      0.73      0.83      1508
        Goal       0.26      0.76      0.38       183

    accuracy                           0.73      1691
   macro avg       0.61      0.75      0.61      1691
weighted avg       0.89      0.73      0.78      1691



### Observations:
- Goal recall 0.76 — model correctly identifies 76% of actual goals
- Goal precision 0.26 — of all shots predicted as Goal, 26% actually are goals (expected given 10.8% base rate)
- The precision/recall trade-off is appropriate for an xG model — we want high recall (catch most goals) at the cost of some false positives
- AUC 0.8183 is a strong result for xG

## Tune Random Forest

Runs `RandomizedSearchCV` with 50 combinations over a 6-parameter Random Forest grid, scored by AUC-ROC via 5-fold stratified CV.

- Random Forest achieved AUC 0.7884 — second best untuned. Tuning it provides a strong non-boosting reference and helps confirm whether gradient boosting is genuinely superior for this task.

- `max_depth=None` means fully grown trees — included alongside depth limits to test whether unlimited depth overfits or helps.


In [16]:
rf_param_grid = {
    'n_estimators':      [100, 200, 300, 500],        # number of trees
    'max_depth':         [None, 5, 10, 15, 20],       # None = fully grown; limited = regularised
    'min_samples_split': [2, 5, 10],                  # min samples to split a node
    'min_samples_leaf':  [1, 2, 4],                   # min samples in a leaf
    'max_features':      ['sqrt', 'log2', None],       # features considered per split
    'class_weight':      ['balanced', 'balanced_subsample'],  # handle imbalance
}

rf = RandomForestClassifier(random_state=42)  # base RF model

search_rf = RandomizedSearchCV(
    rf, rf_param_grid,
    n_iter=50,            # 50 combinations
    cv=cv,                # 5-fold stratified CV
    scoring='roc_auc',    # AUC optimisation
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_rf.fit(X_train_s, y_train)  # run the search

print('Best hyperparameters:')
for k, v in search_rf.best_params_.items():
    print(f'  {k}: {v}')
print()
print(f'Best CV AUC:   {search_rf.best_score_:.4f}')  # CV AUC for best RF params

y_prob_rf_tuned = search_rf.best_estimator_.predict_proba(X_test_s)[:, 1]
y_pred_rf_tuned = search_rf.best_estimator_.predict(X_test_s)
print(f'Test AUC:      {roc_auc_score(y_test, y_prob_rf_tuned):.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred_rf_tuned)*100:.2f}%')
print()
print(classification_report(y_test, y_pred_rf_tuned, target_names=['No Goal', 'Goal']))


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best hyperparameters:
  n_estimators: 300
  min_samples_split: 10
  min_samples_leaf: 2
  max_features: log2
  max_depth: 5
  class_weight: balanced_subsample

Best CV AUC:   0.8120
Test AUC:      0.8176
Test Accuracy: 74.16%

              precision    recall  f1-score   support

     No Goal       0.96      0.74      0.84      1508
        Goal       0.26      0.77      0.39       183

    accuracy                           0.74      1691
   macro avg       0.61      0.75      0.61      1691
weighted avg       0.89      0.74      0.79      1691



### Observations

- Tuned RF AUC 0.8176 — very close to tuned XGBoost (0.8183), only 0.0007 behind
- balanced_subsample (different class weight per tree) chosen over balanced — consistent with Cycle 1 RF tuning result\n- max_depth=5 (shallow) — prevents overfitting, RF with deep trees tends to overfit on small minority classes
- Goal recall 0.77 — marginally higher than XGBoost, but lower precision (0.26)

### Improvement over baseline
- Untuned RF:  AUC 0.7884
- Tuned RF:    AUC 0.8176
- **Gain: +0.0292**

## Tune LightGBM

Runs `RandomizedSearchCV` with 50 combinations over an 8-parameter LightGBM grid, scored by AUC-ROC.

**- LightGBM was the best untuned model (AUC 0.8035). Tuning it is likely to produce the best overall xG model in this cycle.
- `num_leaves` (tree complexity), `min_child_samples` (regularisation), and `scale_pos_weight` at three levels.


In [12]:
lgb_param_grid = {
    'n_estimators':     [100, 200, 300, 500],        # boosting rounds
    'learning_rate':    [0.01, 0.05, 0.1, 0.15],     # shrinkage per tree
    'max_depth':        [3, 4, 5, 6, 7],              # max tree depth
    'num_leaves':       [20, 31, 50, 100],             # main LightGBM complexity parameter
    'subsample':        [0.7, 0.8, 0.9, 1.0],         # row subsampling
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],         # feature subsampling
    'min_child_samples':[10, 20, 30],                  # min samples per leaf — regularisation
    'scale_pos_weight': [scale_pos_weight,             # computed imbalance ratio
                         scale_pos_weight * 0.5,       # softer weight
                         scale_pos_weight * 1.5],      # harder weight
}

total_lgb = 4 * 4 * 5 * 4 * 4 * 4 * 3 * 3  # total combinations
print(f'Total possible LightGBM combinations: {total_lgb:,}')
print(f'We will try: 50 (RandomizedSearch x 5-fold CV = 250 fits)')


Total possible LightGBM combinations: 46,080
We will try: 50 (RandomizedSearch x 5-fold CV = 250 fits)


In [13]:
lgb = LGBMClassifier(random_state=42, verbose=-1)  # base LightGBM model

search_lgb = RandomizedSearchCV(
    lgb, lgb_param_grid,
    n_iter=50,            # 50 random combinations
    cv=cv,                # 5-fold stratified CV
    scoring='roc_auc',    # AUC optimisation
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_lgb.fit(X_train_s, y_train)  # run the search

print('Best hyperparameters:')
for k, v in search_lgb.best_params_.items():
    print(f'  {k}: {v}')
print()
print(f'Best CV AUC:   {search_lgb.best_score_:.4f}')  # best CV AUC

y_prob_lgb_tuned = search_lgb.best_estimator_.predict_proba(X_test_s)[:, 1]  # goal probabilities
y_pred_lgb_tuned = search_lgb.best_estimator_.predict(X_test_s)              # class labels
print(f'Test AUC:      {roc_auc_score(y_test, y_prob_lgb_tuned):.4f}')       # held-out AUC
print(f'Test Accuracy: {accuracy_score(y_test, y_pred_lgb_tuned)*100:.2f}%')


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best hyperparameters:
  subsample: 0.8
  scale_pos_weight: 12.371409028727772
  num_leaves: 100
  n_estimators: 500
  min_child_samples: 10
  max_depth: 3
  learning_rate: 0.01
  colsample_bytree: 0.7

Best CV AUC:   0.8127
Test AUC:      0.8152
Test Accuracy: 64.93%


### Improvement over baseline
- Untuned LGBM:  AUC 0.8035
- Tuned RF:    AUC 0.8152

In [14]:
print('TUNED LIGHTGBM — Full Report')
print(f'AUC-ROC:  {roc_auc_score(y_test, y_prob_lgb_tuned):.4f}')   # primary metric
print(f'Accuracy: {accuracy_score(y_test, y_pred_lgb_tuned)*100:.2f}%')
print()
print(classification_report(y_test, y_pred_lgb_tuned, target_names=['No Goal', 'Goal']))
# Goal recall is key: how many actual goals does the model identify?


TUNED LIGHTGBM — Full Report
AUC-ROC:  0.8152
Accuracy: 64.93%

              precision    recall  f1-score   support

     No Goal       0.98      0.62      0.76      1508
        Goal       0.22      0.87      0.35       183

    accuracy                           0.65      1691
   macro avg       0.60      0.75      0.55      1691
weighted avg       0.89      0.65      0.72      1691



# Full Results Comparison — Tuned vs Untuned

### Key Conclusions

**1. Tuning makes a major difference**
- XGBoost: 0.7871 → 0.8183 (+0.0312)
- Random Forest: 0.7884 → 0.8176 (+0.0292)
- LightGBM: 0.8035 → 0.8152 (+0.0117)

**2. Tuned models beat Logistic Regression**
Both tuned models exceed the LR baseline (0.7963), which led untuned. This confirms that while LR is a strong default for xG, gradient boosting surpasses it with proper tuning.

**3. Accuracy is not the metric**

All tuned models show accuracy around 73-74% — lower than the 89% dummy. This is correct behaviour: the models are trading some false negatives (predicting No Goal when it is Goal) for high recall on actual goals.

# Save Best Model

Identifies the highest-AUC tuned model, then saves three artefacts: the model itself, the fitted scaler, and the feature column list.

In [17]:
# Identify the best model by AUC score
xgb_auc = roc_auc_score(y_test, y_prob_xgb_tuned)  # tuned XGBoost AUC
rf_auc  = roc_auc_score(y_test, y_prob_rf_tuned)    # tuned RF AUC
lgb_auc = roc_auc_score(y_test, y_prob_lgb_tuned)   # tuned LightGBM AUC

best_auc   = max(xgb_auc, rf_auc, lgb_auc)          # highest AUC wins
best_model = {xgb_auc: search_xgb.best_estimator_, rf_auc: search_rf.best_estimator_, lgb_auc: search_lgb.best_estimator_}[best_auc]
best_name  = {xgb_auc: 'XGBoost Tuned', rf_auc: 'Random Forest Tuned', lgb_auc: 'LightGBM'}[best_auc]

print(f'Saving: {best_name} (AUC={best_auc:.4f})')

# Save three artefacts needed by the API to make predictions
joblib.dump(best_model,            str(Paths.C2_MODEL))     # trained model
joblib.dump(scaler,                str(Paths.C2_SCALER))    # fitted scaler (same transform must be applied to API input)
joblib.dump(list(X_train.columns), str(Paths.C2_FEATURES))  # column order (API must send features in this order)

print('Saved: ../../models/cycle2_best_model.pkl')
print('Saved: ../../models/cycle2_scaler.pkl')
print('Saved: ../../models/cycle2_feature_cols.pkl')


Saving: XGBoost Tuned (AUC=0.8183)
Saved: ../../models/cycle2_best_model.pkl
Saved: ../../models/cycle2_scaler.pkl
Saved: ../../models/cycle2_feature_cols.pkl


- Three artefacts saved: model, scaler, feature column list
- Same modular pattern as Cycle 1 — each component can be updated independently
- Features: ['X', 'Y', 'Distance', 'Angle', 'Left_Foot', 'Right_Foot', 'Header', 'First_Half', 'Player_Rank']
- The API will load these three files at startup to serve xG predictions